# Revacc T4 Backend — Complete Pipeline with Local ESMFold

**Architecture:** T4 GPU = backend, Vercel = frontend, ngrok = tunnel

All thresholds match Barazesh et al. 2024 (Nature Sci Reports). No mock data.

1. Install all dependencies (BLAST+, Python 3.10, PyTorch CUDA, OpenFold)
2. Download ESMFold 3B weights (2.7 GB)
3. Build OpenFold with C++17 patch for T4
4. Clone repo, configure paper thresholds
5. Start FastAPI backend + ngrok tunnel
6. Run full 50-step pipeline

In [ ]:
#@title 1. Set your ngrok authtoken
# Get from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTHTOKEN = ""  #@param {type:"string"}

#@title Pipeline configuration
TAXON_ID = "99287"  #@param {type:"string"}  # Streptococcus agalactiae
PATHOGEN_NAME = "Streptococcus agalactiae"  #@param {type:"string"}

import os
os.environ["NGROK_AUTHTOKEN"] = NGROK_AUTHTOKEN
print(f"Taxon: {TAXON_ID} ({PATHOGEN_NAME})")

In [ ]:
#@title 2. Install system deps + Python 3.10 + BLAST+ (run once)
!apt-get update -qq && apt-get install -y -qq \
  blast+ ncbi-blast+ \
  python3.10 python3.10-venv python3.10-dev python3.10-distutils \
  git wget curl > /dev/null 2>&1

!blastp -version 2>&1 | head -1
!python3.10 --version

In [ ]:
#@title 3. Clone repo + create Python 3.10 venv + install all deps
import subprocess, os

WORKDIR = "/content/Revacc"
VENV = "/content/esmfold-env"

# Clone repo
if not os.path.exists(WORKDIR):
    !git clone https://github.com/umeshdahiya15/Revacc.git {WORKDIR}
else:
    os.chdir(WORKDIR)
    !git pull origin main

# Create venv
if not os.path.exists(VENV):
    !python3.10 -m venv {VENV}

# Install PyTorch CUDA + backend deps + ngrok
!{VENV}/bin/pip install --upgrade pip 'setuptools<81' wheel > /dev/null 2>&1
!{VENV}/bin/pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 > /dev/null 2>&1
!{VENV}/bin/pip install -r {WORKDIR}/backend/requirements.txt > /dev/null 2>&1
!{VENV}/bin/pip install numpy==1.24.2 matplotlib==3.7.0 pyngrok > /dev/null 2>&1

# Verify
!{VENV}/bin/python -c "import torch; print(f'PyTorch {torch.__version__}, CUDA={torch.cuda.is_available()}, GPU={torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"N/A\"}')"

In [ ]:
#@title 4. Download ESMFold 3B weights (2.7 GB)
WEIGHTS_DIR = os.path.expanduser("~/.cache/torch/hub/checkpoints")
WEIGHTS_FILE = os.path.join(WEIGHTS_DIR, "esmfold_3B_v1.pt")

if not os.path.exists(WEIGHTS_FILE):
    os.makedirs(WEIGHTS_DIR, exist_ok=True)
    !wget -q --show-progress -O {WEIGHTS_FILE} \
      https://dl.fbaipublicfiles.com/fair-esm/models/esmfold_3B_v1.pt

size_gb = os.path.getsize(WEIGHTS_FILE) / (1024**3)
print(f"ESMFold weights: {size_gb:.1f} GB at {WEIGHTS_FILE}")

In [ ]:
#@title 5. Build OpenFold with C++17 patch (required for PyTorch 2.x)
OPENFOLD_DIR = "/content/openfold"

if not os.path.exists(OPENFOLD_DIR):
    !git clone --filter=blob:none https://github.com/aqlaboratory/openfold.git {OPENFOLD_DIR}
    os.chdir(OPENFOLD_DIR)
    !git checkout 4b41059

# CRITICAL: Patch C++14 -> C++17 for PyTorch 2.x
!find {OPENFOLD_DIR} -name 'setup.py' -exec sed -i 's/-std=c++14/-std=c++17/g' {} +
!find {OPENFOLD_DIR} -name '*.cpp' -exec sed -i 's/-std=c++14/-std=c++17/g' {} + 2>/dev/null
!find {OPENFOLD_DIR} -name '*.cu' -exec sed -i 's/-std=c++14/-std=c++17/g' {} + 2>/dev/null

# Install OpenFold deps + older PyTorch Lightning
!{VENV}/bin/pip install -r {OPENFOLD_DIR}/requirements.txt > /dev/null 2>&1
!{VENV}/bin/pip install pytorch-lightning==1.9.5 > /dev/null 2>&1

# Build CUDA extensions (takes ~3-5 min)
print("Building OpenFold CUDA extensions...")
!cd {OPENFOLD_DIR} && {VENV}/bin/python setup.py install 2>&1 | tail -5
print("OpenFold build complete.")

In [ ]:
#@title 6. Verify ESMFold runs on T4
verify_script = '''
import torch
print(f"CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0)}")
model = torch.hub.load("facebookresearch/esm:main", "esmfold_3B_v1")
model = model.eval().cuda()
print(f"ESMFold on: {next(model.parameters()).device}")
pdb = model.infer_pdb("MKFLILLFNILCLFPVLAADNHGVSLQGFNKENYEKFDKARLENGITYDSIMYSGRDFNE")
print(f"Test fold: {len(pdb)} chars PDB — ESMFold WORKING ON T4!")
'''
with open('/tmp/verify_esmfold.py', 'w') as f:
    f.write(verify_script)
!{VENV}/bin/python /tmp/verify_esmfold.py

In [ ]:
#@title 7. Configure paper-aligned thresholds + start backend + ngrok
import subprocess, time, os, json, signal

WORKDIR = "/content/Revacc"
VENV = "/content/esmfold-env"
BACKEND_PORT = 8000

# Kill any existing processes
!pkill -f 'uvicorn.*{BACKEND_PORT}' 2>/dev/null; true
time.sleep(1)

# Paper-aligned environment variables
env = os.environ.copy()
env.update({
    "PATH": f"{VENV}/bin:{env.get('PATH', '')}",
    "PYTHONPATH": WORKDIR,
    # Paper thresholds (Barazesh et al. 2024)
    "CDHIT_THRESHOLD": "0.80",
    "DEG_IDENTITY_THRESHOLD": "20",
    "DEG_EVALUE_THRESHOLD": "1e-5",
    "ALGPRED_THRESHOLD": "0.321",
    "VAXIJEN_THRESHOLD": "0.50",
    "VFDB_EVALUE_THRESHOLD": "1e-4",
    "VFDB_BITSCORE_THRESHOLD": "100",
    "VFDB_IDENTITY_THRESHOLD": "30",
    "HUMAN_HOMOLOGY_IDENTITY": "0.30",
    "HOMOLOGY_EVALUE": "1e-4",
    "MHC1_PERCENTILE": "2.0",
    "MHC2_PERCENTILE": "2.0",
    "IL4_THRESHOLD": "0.2",
    "IL10_THRESHOLD": "-0.3",
    "BCELL_THRESHOLD": "0.5",
    "MEV_STRUCTURE_PROVIDER": "esmfold",
    "MEV_CORS_ORIGINS": "*",
    "MEV_STEP_TICK_MS": "300",
    "MEV_BLAST_DB_CACHE": "/content/blast_dbs",
    "MEV_VFDB_CACHE": "/content/blast_dbs/vfdb",
})
os.makedirs("/content/blast_dbs", exist_ok=True)

# Start backend
backend_proc = subprocess.Popen(
    [f"{VENV}/bin/python", "-m", "uvicorn", "app.main:app",
     "--host", "0.0.0.0", "--port", str(BACKEND_PORT)],
    cwd=os.path.join(WORKDIR, "backend"),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Wait for backend
import urllib.request
for i in range(30):
    time.sleep(1)
    try:
        urllib.request.urlopen(f"http://localhost:{BACKEND_PORT}/api/health", timeout=2)
        print(f"Backend started (PID: {backend_proc.pid})")
        break
    except Exception:
        if i == 29:
            print("Backend failed to start!")

# Start ngrok
public_url = None
if NGROK_AUTHTOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    ngrok.kill()
    tunnel = ngrok.bind(BACKEND_PORT)
    public_url = tunnel.public_url
    print(f"\n{'='*60}")
    print(f"BACKEND TUNNEL: {public_url}")
    print(f"{'='*60}")
    print(f"\nSet in Vercel: NEXT_PUBLIC_API_URL={public_url}")
else:
    print(f"\nBackend at http://localhost:{BACKEND_PORT}")
    print("Set NGROK_AUTHTOKEN to expose to Vercel")

In [ ]:
#@title 8. Run the full 50-step pipeline
import urllib.request, json, time

BASE = f"http://localhost:{BACKEND_PORT}"

# Create job
job_data = json.dumps({
    "taxonId": TAXON_ID,
    "pathogenName": PATHOGEN_NAME,
    "realTools": True,
}).encode("utf-8")
req = urllib.request.Request(f"{BASE}/api/jobs", data=job_data,
                             headers={"Content-Type": "application/json"},
                             method="POST")
with urllib.request.urlopen(req) as resp:
    job = json.loads(resp.read())
    job_id = job["id"]
print(f"Created job: {job_id}")

# Start pipeline
req = urllib.request.Request(f"{BASE}/api/jobs/{job_id}/start", method="POST")
with urllib.request.urlopen(req) as resp:
    print(f"Pipeline started: {resp.status}")

# Poll until complete
while True:
    try:
        req = urllib.request.Request(f"{BASE}/api/jobs/{job_id}")
        with urllib.request.urlopen(req) as resp:
            status = json.loads(resp.read())
        ps = status.get("status", "unknown")
        step = status.get("currentStep", "")
        phase = status.get("currentPhase", 0)
        if ps in ("completed", "failed", "error"):
            print(f"\nPipeline {ps}!")
            break
        print(f"Phase {phase}, Step {step} [{ps}]", end="\r")
        time.sleep(15)
    except Exception as e:
        time.sleep(5)

# Print results
print(f"\n\n{'='*60}")
print("RESULTS")
print(f"{'='*60}")
funnel = status.get("funnel", {})
for k, v in funnel.items():
    if isinstance(v, dict):
        print(f"  {k}: {v.get('count', v)}")
    else:
        print(f"  {k}: {v}")

for phase in status.get("phases", []):
    for step in phase.get("steps", []):
        if step.get("id") == "9-2" and step.get("result"):
            r = step["result"]
            print(f"\nMEV: {r.get('mev_length', 0)} aa")
            print(f"Sequence: {r.get('sequence', '')[:100]}...")
        if step.get("id") == "11-2" and step.get("result"):
            r = step["result"]
            print(f"\nStructure: {r.get('provider')} / {r.get('method')}")